<a href="https://colab.research.google.com/github/ryankrismer/EM_field_solver/blob/main/EM_field_solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Electromagnetic field solver

In [21]:
import numpy as np
import tqdm
import sys

# Set threshold to infinity or system maximum
np.set_printoptions(threshold=sys.maxsize)

## General functions

In [ ]:
def find_dist(r, r_ref):
  """
  Function to calculate the spatial distance between 2 coordinates

  Parameters:
    r:      [x, y, z] (m) (1D numpy array)
    r_ref:  [x_ref, y_ref, z_ref] (m) (1D numpy array of floats)

  Returns: Distance between coordinates (float)
  """
  return np.sqrt((r[0] - r_ref[0]) ** 2 + (r[1] - r_ref[1]) ** 2 + (r[2] - r_ref[2]) ** 2)

In [ ]:
def find_r(x, y, z):
  """
  Function to find spherical radial coordinate value, given a particular cartesian position

  Parameters:
    x: x coordinate value (m) (float)
    y: y coordinate value (m) (float)
    z: z coordinate value (m) (float)

  Returns: Value of spherical radial coordinate (m) (float)
  """
  return find_dist([x, y, z], [0.0, 0.0, 0.0])

In [ ]:
def find_theta(z, r):
  """
  Function to find spherical polar coordinate value, given a particular cartesian position

  Parameters:
    z: z coordinate value (m) (float)
    r: Spherical radial coordinate value (m) (float)

  Returns: Value of spherical polar coordinate (rad) (float)
  """
  return np.arccos(z / r)

In [ ]:
def find_Delta_r(x, y, z):
  """
  Function to find value of spherical radial coordinate differential for particular cartesian position

  Parameters:
    x: x coordinate value (m) (float)
    y: y coordinate value (m) (float)
    z: z coordinate value (m) (float)

  Returns: Value of spherical radial coordinate differential (m) (float)

  Note: Assumes a cartesian differential corresponding to source lattice
  """
  return Delta_L_src * np.abs(x + y + z) / find_r(x, y, z)

## Setting sources and lattices

In [23]:
# Defining constants and parameters

# E&M constants
c = 299792458                         # Speed of light in vacuum (m * s^-1)
epsilon_0 = 8.8541878188 * 10 ** -12  # Vacuum electric permissivity (F * m^-1)
mu_0 = 1.25663706127 * 10 ** -6       # Vacuum magnetic permeability (N * A^-2)

###############################################################################

# Pulsar parameters
R = 10.0 * 10 ** 3    # Radius (m)
omega = 2 * np.pi     # Rotational speed (rad * s^-1)
chi = np.pi / 4       # Angle between magnetic dipole axis and rotational axis z (rad)
cos_chi = np.cos(chi) # Cosine of angle between magnetic dipole axis and rotational axis z (rad)
sin_chi = np.sin(chi) # Sine of angle between magnetic dipole axis and rotational axis z (rad)
R1 = 10 ** 8 / mu_0   # Arbitrary function of spherical radial coordinate; same order of magnitude as surface H field (A * m^-1)
R3 = 10 ** 8 / mu_0   # Arbitrary function of spherical radial coordinate; same order of magnitude as surface H field (A * m^-1)

################################################################################################################################

# General lattice parameters pt 1
N_plot = 40 ** 4  # Number of observation points to be plotted

##############################################################

# Source lattice parameters
L_src_out = R               # Outer boundary of source lattice (m)
Delta_L_src_goal = 0.1 * R  # Desired distance between adjacent source spatial coordinate values; should be at most 0.1R (m)

n_src_max = 2 * (int(np.round(np.abs(L_src_out + L_src_out)) / Delta_L_src_goal)) # Size of each source spatial coordinate axis
print(f"n_src_max is {n_src_max}")
Delta_L_src = (L_src_out + L_src_out) / n_src_max                                 # Actual distance between adjacent source spatial coordinate values (m)
print(f"Delta_L_src is {Delta_L_src}")

# Ensuring L_in is an integer number of grid spacings
L_in_goal = R / np.sqrt(3.0)
L_in = int(L_in_goal / Delta_L_src) * Delta_L_src # Inner boundary of source lattice (m)
print(f"L_in is {L_in}, {L_in / Delta_L_src} * Delta_L_src")

#########################################################################################################################################################

# Setting source spatial lattice points
P_src_tbxy = np.arange(-L_src_out, L_src_out + Delta_L_src, Delta_L_src)  # x and y coordinate values for source top and bottom lattices (m)

P_src_tz_1x_2y = np.arange(L_in, L_src_out + Delta_L_src, Delta_L_src)    # z coordinate values for source top lattice,
                                                                          # x coordinate values for source lattice 1,
                                                                          # y coordinate values for source lattice 2 (m)

P_src_bz_3x_4y = np.arange(-L_src_out, -L_in + Delta_L_src, Delta_L_src)  # z coordinate values for source bottom lattice,
                                                                          # x coordinate values for source lattice 3,
                                                                          # y coordinate values for source lattice 4 (m)

P_src_sz = np.arange(-L_in, L_in, Delta_L_src)                            # z coordinate values for source side lattices (m)

P_src_2x_3y = np.arange(-L_src_out, L_in, Delta_L_src)                    # x coordinate values for source lattice 2,
                                                                          # y coordinate values for source lattice 3 (m)

P_src_4x_1y = np.arange(-L_in, L_src_out + Delta_L_src, Delta_L_src)      # x coordinate values for source lattice 4,
                                                                          # y coordinate values for source lattice 1 (m)

############################################################################################################################################

# Observation lattice parameters
L_obs_out = 10 * R                                                                                              # Outer boundary of observation lattice (m)
N_obs = 2 * int(np.round(((3 * N_plot ** (3 / 4) / (4 * np.pi)) / (1 - 1 / (L_obs_out / R) ** 3)) ** (1 / 3)))  # Size of each observation spatial coordinate axis
print(f"N_obs is {N_obs}")
Delta_L_obs = (L_obs_out - L_in) / N_obs                                                                        # Distance between adjacent observation spatial coordinate values (m)

#####################################################################################################################################################################################

# General lattice parameters pt 2
Delta_t = Delta_L_src / c     # Difference between consecutive time coordinate values (s)
N_t = int(N_plot ** (1 / 4))  # Size of time coordinate axis
print(f"N_t is {N_t}")

#########################################################################################

# Time lattice points
T = np.linspace(0.0, Delta_t * (N_t - 1), N_t)  # Time axis (s)

L_in is 5773.502691896258
n_src is 36
Delta_L_src is 117.40270300288174
N_obs is 50
N_t is 40


### Functions

In [ ]:
def find_src_params(t,
                    x,
                    y,
                    z,
                    r,
                    omega,
                    cos_chi,
                    sin_chi,
                    R3,
                    R1 = None
                   ):
  """
  Function to find values of source coordinates and functions from particular cartesian position

  Parameters:
    t:        Time coordinate value (s) (float)
    x:        x coordinate value (m) (float)
    y:        y coordinate value (m) (float)
    z:        z coordinate value (m) (float)
    r:        Spherical radial coordinate value (m) (float)
    omega:    Rotational speed of pulsar (rad * s^-1) (float)
    cos_chi:  Cosine of angle between pulsar's magnetic dipole axis and rotational axis z (dimensionless) (float)
    sin_chi:  Sine of angle between pulsar's magnetic dipole axis and rotational axis z (dimensionless) (float)
    R3:       Arbitrary function of spherical radial coordinate; same order of magnitude as surface H field (A * m^-1) (float)
    R1:       Arbitrary function of spherical radial coordinate; same order of magnitude as surface H field (A * m^-1) (float)

  Returns:
    cos_phi:    Cosine of spherical azimuthal coordinate (rad) (float)
    sin_phi:    Sine of spherical azimuthal coordinate (rad) (float)
    cos_lambda: Cosine of lambda coordinate (dimensionless) (float)
    theta:      Value of spherical polar coordinate (rad) (float)
    cos_theta:  Cosine of spherical polar coordinate (dimensionless) (float)
    sin_theta:  Sine of spherical polar coordinate (dimensionless) (float)
    R2:         Value of arbitrary function R2, given R1 (A * m^-1) (float)
    C:          Value of C function (dimensionless) (float)
    D:          Value of D function (A * m^-1) (float)
    F:          Value of F function (dimensionless) (float)
    G:          Value of G function (A * m^-1) (float)

  Note: the current form assumes R1 is constant
  """
  phi = np.arctan2(y, x)
  cos_phi = np.cos(phi)
  sin_phi = np.sin(phi)

  lambda_coord = phi - omega * t
  cos_lambda = np.cos(lambda_coord)

  theta = find_theta(z, r)
  cos_theta = np.cos(theta)
  sin_theta = np.sin(theta)

  C = cos_chi * sin_theta - sin_chi * cos_theta * cos_lambda

  psi = np.arccos(cos_chi * cos_theta + sin_chi * sin_theta * cos_lambda)
  S3 = np.cos(psi)
  D = R3 * S3 / np.sin(psi)

  F = sin_chi * np.sin(lambda_coord)

  # Only calculating R2 and G if necessary
  if R1 is None:
    R2 = None
    G = None
  else:
    R2 = -R1  # Will need to be modified if R1 isn't constant
    G = R2 - 1 / 2 * R1

  return cos_phi, sin_phi, cos_lambda, theta, cos_theta, sin_theta, R2, C, D, F, G

In [ ]:
def src(t,
        x,
        y,
        z,
        omega = 2 * np.pi,
        R1 = 10 ** 8 / mu_0,
        R3 = 10 ** 8 / mu_0,
        cos_chi = np.sqrt(2.0) / 2,
        sin_chi = np.sqrt(2.0) / 2
       ):
  """
  Function to find charge and current distributions for vacuum retarded dipole model of a pulsar

  Parameters:
    t:        Time coordinate value (s) (float)
    x:        x coordinate value (m) (float)
    y:        y coordinate value (m) (float)
    z:        z coordinate value (m) (float)
    omega:    Rotational speed of pulsar (rad * s^-1) (float)
    R1:       Arbitrary function of spherical radial coordinate; same order of magnitude as surface H field (A * m^-1) (float)
    R3:       Arbitrary function of spherical radial coordinate; same order of magnitude as surface H field (A * m^-1) (float)
    cos_chi:  Cosine of angle between pulsar's magnetic dipole axis and rotational axis z (rad) (float)
    sin_chi:  Sine of angle between pulsar's magnetic dipole axis and rotational axis z (rad) (float)

  Returns:  Surface charge density and surface current density at source position (t, x, y, z)
            (form [sigma, K_x, K_y, K_z]) (sigma is C * m^-2, K_i is A * m^-1) (each is float)
  """
  r = find_r(x, y, z)

  cos_phi, sin_phi, cos_lambda, theta, cos_theta, sin_theta, R2, C, D, F, G = find_src_params(t,
                                                                                              x,
                                                                                              y,
                                                                                              z,
                                                                                              r,
                                                                                              omega,
                                                                                              cos_chi,
                                                                                              sin_chi,
                                                                                              R3,
                                                                                              R1
                                                                                             )

  # Sigma
  sigma_term_1a = cos_chi * (3 * np.cos(2 * theta) + 1)
  sigma_term_1b = 3 * sin_chi * np.sin(2 * theta) * cos_lambda

  sigma_term_1 = 1 / 4 * R1 * (sigma_term_1a + sigma_term_1b)
  sigma_term_2 = (R2 * C - D * F) * sin_theta

  sigma = -omega * R / c ** 2 * (sigma_term_1 + sigma_term_2)

  # K_theta and K_phi
  K_theta = G * F + D * C
  K_phi = -G * C + D * F

  # K_x
  K_x_term_1 = K_theta * cos_theta * cos_phi
  K_x_term_2 = -K_phi * sin_phi

  K_x = K_x_term_1 + K_x_term_2

  # K_y
  K_y_term_1 = K_theta * cos_theta * sin_phi
  K_y_term_2 = K_phi * cos_phi

  K_y = K_y_term_1 + K_y_term_2

  # K_z
  K_z = -K_theta * sin_theta

  return [sigma, K_x, K_y, K_z]

### Creating source lattice positions and sources

In [ ]:
# Creating sources
n_src_tbxy = np.size(P_src_tbxy)
n_src_tz_1x_2y = np.size(P_src_tz_1x_2y)
n_src_bz_3x_4y = np.size(P_src_bz_3x_4y)
n_src_4x_1y = np.size(P_src_4x_1y)
n_src_sz = np.size(P_src_sz)
n_src_2x_3y = np.size(P_src_2x_3y)

sigma_t = np.full((N_t, n_src_tbxy, n_src_tbxy, n_src_tz_1x_2y), np.nan)
K_x_t = sigma_t
K_y_t = sigma_t
K_z_t = sigma_t

sigma_b = np.full((N_t, n_src_tbxy, n_src_tbxy, n_src_bz_3x_4y), np.nan)
K_x_b = sigma_b
K_y_b = sigma_b
K_z_b = sigma_b

sigma_1 = np.full((N_t, n_src_tz_1x_2y, n_src_4x_1y, n_src_sz), np.nan)
K_x_1 = sigma_1
K_y_1 = sigma_1
K_z_1 = sigma_1

sigma_2 = np.full((N_t, n_src_2x_3y, n_src_tz_1x_2y, n_src_sz), np.nan)
K_x_2 = sigma_2
K_y_2 = sigma_2
K_z_2 = sigma_2

sigma_3 = np.full((N_t, n_src_bz_3x_4y, n_src_2x_3y, n_src_sz), np.nan)
K_x_3 = sigma_3
K_y_3 = sigma_3
K_z_3 = sigma_3

sigma_4 = np.full((N_t, n_src_4x_1y, n_src_bz_3x_4y, n_src_sz), np.nan)
K_x_4 = sigma_4
K_y_4 = sigma_4
K_z_4 = sigma_4

# Packing lattice stuff up
n_src_x_arr = [n_src_tbxy, n_src_tbxy, n_src_tz_1x_2y, n_src_2x_3y, n_src_bz_3x_4y, n_src_4x_1y]
P_src_x_arr = [P_src_tbxy, P_src_tbxy, P_src_tz_1x_2y, P_src_2x_3y, P_src_bz_3x_4y, P_src_4x_1y]

n_src_y_arr = [n_src_tbxy, n_src_tbxy, n_src_4x_1y, n_src_tz_1x_2y, n_src_2x_3y, n_src_bz_3x_4y]
P_src_y_arr = [P_src_tbxy, P_src_tbxy, P_src_4x_1y, P_src_tz_1x_2y, P_src_2x_3y, P_src_bz_3x_4y]

n_src_z_arr = [n_src_tz_1x_2y, n_src_bz_3x_4y, n_src_sz, n_src_sz, n_src_sz, n_src_sz]
P_src_z_arr = [P_src_tz_1x_2y, P_src_bz_3x_4y, P_src_sz, P_src_sz, P_src_sz, P_src_sz]

sigma_arr = [sigma_t, sigma_b, sigma_1, sigma_2, sigma_3, sigma_4]
K_x_arr = [K_x_t, K_x_b, K_x_1, K_x_2, K_x_3, K_x_4]
K_y_arr = [K_y_t, K_y_b, K_y_1, K_y_2, K_y_3, K_y_4]
K_z_arr = [K_z_t, K_z_b, K_z_1, K_z_2, K_z_3, K_z_4]
label_arr = ["t", "b", "1", "2", "3", "4"]

# Setting each source point
for l in range(6):

  # Unpacking lattice stuff
  label = label_arr[l]
  n_src_x = n_src_x_arr[l]
  n_src_y = n_src_y_arr[l]
  n_src_z = n_src_z_arr[l]

  P_src_x = P_src_x_arr[l]
  P_src_y = P_src_y_arr[l]
  P_src_z = P_src_z_arr[l]

  sigma = sigma_arr[l]
  K_x = K_x_arr[l]
  K_y = K_y_arr[l]
  K_z = K_z_arr[l]

  for t in tqdm.tqdm(range(N_t)):
    for x in range(n_src_x):
      for y in range(n_src_y):
        for z in range(n_src_z):
          t_i = T[t]
          x_i = P_src_x[x]
          y_i = P_src_y[y]
          z_i = P_src_z[z]
          r = find_r(x_i, y_i, z_i)

          # Finding the maximum uncertainty in r
          r_errs = []

          for i in range(2):
            for j in range(2):
              for k in range(2):
                x_corner = x_i + (-1) ** i * Delta_L_src / 2
                y_corner = y_i + (-1) ** j * Delta_L_src / 2
                z_corner = z_i + (-1) ** k * Delta_L_src / 2
                r_errs.append(np.abs(r - find_r(x_corner, y_corner, z_corner)))

          r_err = np.max(r_errs)

          if np.abs(r - R) <= r_err:  # Dirac delta function condition
            src_i = find_Delta_r(x_i, y_i, z_i) * np.array(src(t_i, x_i, y_i, z_i, omega, R1, R3, cos_chi, sin_chi))
            sigma[t, x, y, z] = src_i[0]
            K_x[t, x, y, z] = src_i[1]
            K_y[t, x, y, z] = src_i[2]
            K_z[t, x, y, z] = src_i[3]
          else:
            continue

  np.save(f"sigma_{label}.npy", sigma)
  np.save(f"K_x_{label}.npy", K_x)
  np.save(f"K_y_{label}.npy", K_y)
  np.save(f"K_z_{label}.npy", K_z)

## Finding potentials from sources

### Functions

In [ ]:
def causally_related(dist_src, t_i_src, t_i_obs):
  """
  Function to evaluate whether source point is causally related to observation point

  Parameters:
    dist_src: Distance between source and observation points (m) (float)
    t_i_src:  Value of time for source point (s) (float)
    t_i_obs:  Value of time for observation point (s) (float)
    L_err:    Uncertainty due to discreteness of spatial lattice (m) (float)

  Returns: True if source point is causally related to observation point, False otherwise (boolean)
  """
  t_r_err = (Delta_t + np.sqrt(3) / c * (Delta_L_src + Delta_L_obs)) / 2

  return np.abs(t_i_obs - dist_src / c - t_i_src) <= t_r_err

In [ ]:
def Delta_theta(x, y, z, r):
  """
  Function to find value of spherical polar coordinate differential for particular cartesian position

  Parameters:
    x: x coordinate value (m) (float)
    y: y coordinate value (m) (float)
    z: z coordinate value (m) (float)
    r: spherical radial coordinate value (m) (float)

  Returns: Value of spherical polar coordinate differential (rad) (float)

  Note: Assumes a cartesian differential corresponding to source lattice
  """
  return Delta_L_src / np.sqrt(x ** 2 + y ** 2) * np.abs(z * (x + y + z) / r ** 2 - 1)

In [ ]:
def Delta_phi(x, y):
  """
  Function to find value of spherical azimuthal coordinate differential for particular cartesian position

  Parameters:
    x: x coordinate value (m) (float)
    y: y coordinate value (m) (float)

  Returns: Value of spherical polar coordinate differential (rad) (float)

  Note: Assumes a cartesian differential corresponding to source lattice
  """
  return Delta_L_src * np.abs(x - y) / (x ** 2 + y ** 2)

In [ ]:
def Delta_A(x, y, z):
  """
  Function to find value of spherical surface area differential for particular cartesian position

  Parameters:
    x: x coordinate value (m) (float)
    y: y coordinate value (m) (float)
    z: z coordinate value (m) (float)

  Returns: Value of spherical surface area differential (m^2) (float)

  Note: Assumes a cartesian differential corresponding to source lattice
  """
  r = find_r(x, y, z)
  return Delta_theta(x, y, z, r) * Delta_phi(x, y) * r ** 2 * np.sin(find_theta(z, r))

In [ ]:
def find_src_sum(src, Delta_f):
  """
  Function to find source sums for potential functions

  Parameters:
    src:      Source distribution (4D numpy array of floats)
    Delta_f:  Differential function to evaluate: Delta_A() for scalar potential and Delta_r for vector potential (function)

  Returns: Sums of source contributions for each observation point (4D numpy array of floats)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  src_sum = np.full((N_t, N_obs, N_obs, N_obs), np.nan) # Creating 4D array w/ set number of values per observation coordinate

  # Calculating and setting values for each plottable observation point
  for t_obs in tqdm.tqdm(range(N_t)):
    t_i_obs = T[t_obs]

    for x_obs in tqdm.tqdm(range(N_obs)):
      x_i_obs = P_obs[x_obs]

      for y_obs in range(N_obs):
        y_i_obs = P_obs[y_obs]
        r_x_y = find_r(x_i_obs, y_i_obs, 0.0) # Distance of (x, y) from origin (m)

        if r_x_y > L_obs_out: # True if (x, y) line is beyond plotting region
          if y_i_obs > 0.0:   # True if all future y values will also be beyond plotting region
            break

          continue  # Skip this y value but continue iterating through y

        for z_obs in range(N_obs):
          valid_obs_point = False               # Default to not summing up source contributions
          z_i_obs = P_obs[z_obs]
          r = find_r(x_i_obs, y_i_obs, z_i_obs) # Distance from origin (m)

          if r > L_obs_out or r <= R:  # Ensuring we only save values we're going to plot
            break

          src_i = []

          # Iterating through all source points that occur at an earlier time than observation time
          for t_src in range(t_obs + 1):
            t_i_src = T[t_src]

            for x_src in range(n_src_x_arr[0]):
              x_i_src = P_src_x[0][x_src]

              for y_src in range(n_src_y_arr[0]):
                y_i_src = P_src_y[0][y_src]

                for z_src in range(n_src_z_arr[0]):
                  if np.isnan(src[t_src, x_src, y_src, z_src]): # Ensuring we don't check empty points
                    continue

                  z_i_src = P_src_z_arr[0][z_src]
                  dist_src = find_dist([x_i_obs, y_i_obs, z_i_obs], [x_i_src, y_i_src, z_i_src])  # Distance from src (m)

                  # Only contributions are source points causally related to observation points
                  if causally_related(dist_src, t_i_src, t_i_obs):
                    valid_obs_point = True
                    src_i.append(Delta_f(x_i_src, y_i_src, z_i_src) * src[t_src, x_src, y_src, z_src] / dist_src)

          if valid_obs_point:
            src_sum[t_obs, x_obs, y_obs, z_obs] = np.sum(src_i) # Adding up all the source contributions

  return src_sum

In [ ]:
def find_Phi(sigma):
  """
  Function to find scalar potential from charge distribution

  Parameter:
    sigma: charge distribution (collection of surface density segments) (C * m^-2) (4D numpy array of floats)

  Returns: scalar potential (V) (4D numpy array of floats)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  return Delta_t / (4 * np.pi * epsilon_0) * find_src_sum(sigma, Delta_A)

In [ ]:
def find_A_i(K_i):
  """
  Function to find component of vector potential from current distribution

  Parameter:
    K_i: i component of current distribution (collection of surface density segments) (A * m^-1) (4D numpy array of floats)

  Returns: i component of vector potential (T * m) (4D numpy array of floats)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  return mu_0 * Delta_t * Delta_L_src / (4 * np.pi) * find_src_sum(K_i, find_Delta_r)

### Finding potentials

In [ ]:
# Setting observation lattice points
P_obs = np.linspace(-L_obs_out, L_obs_out, N_obs) # Positional coordinate values for each observation spatial axis (m)

In [ ]:
# sigma_1 = np.load("sigma_1.npy")
Phi_t = find_Phi(sigma_t)
np.save(f"Phi_t.npy", Phi_t)

  0%|          | 0/40 [00:01<?, ?it/s]


KeyboardInterrupt: 

## Finding fields from potentials

### Functions

In [ ]:
def find_E_i(Phi, A, i):
  """
  Function to find E field component from potentials Phi and A_i

  Parameters:
    Phi:  scalar potential (V) (4D numpy array of floats)
    A:    [A_x, A_y, A_z] components of vector potential (T * m) (each is 4D numpy array of floats)
    i:    field component desired

  Returns: i component of E field (N * C^-1) (4D numpy array of floats)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  if i != 1 and i != 2 and i != 3:
    raise ValueError("i value not allowed. i must be a spatial index within [1, 2, 3]")

  # Finding A component based on index value
  A_i = A[i - 1]

  return -np.gradient(Phi, axis = i) - np.gradient(A_i, axis = 0)

In [ ]:
def find_B_i(A, i):
  """
  Function to find B field component from potential A

  Parameters:
    A: [A_x, A_y, A_z] components of vector potential (T * m) (each is 4D numpy array of floats)
    i: field component desired

  Returns: i component of B field (T) (4D numpy array of floats)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  # Finding index values to ensure cyclicity
  if i == 1:
    j = 2
    k = 3
  elif i == 2:
    j = 3
    k = 1
  elif i == 3:
    j = 1
    k = 2
  else:
    raise ValueError("i value not allowed. i must be a spatial index within [1, 2, 3]")

  # Finding A components based on index values
  A_j = A[j - 1]
  A_k = A[k - 1]

  return np.gradient(A_k, axis = j) - np.gradient(A_j, axis = k)

### Tests

In [ ]:
# # Test of find_E_i() w/ test phi
# A_x = np.random.rand(N_t, N_obs, N_obs, N_obs)
# np.save(f"A_x_test_{n_src}.npy", A_x)
# A_y = np.random.rand(N_t, N_obs, N_obs, N_obs)
# np.save(f"A_y_test_{n_src}.npy", A_y)
# A_z = np.random.rand(N_t, N_obs, N_obs, N_obs)
# np.save(f"A_z_test_{n_src}.npy", A_z)
# E_x = find_E_i(Phi, [A_x, A_y, A_z], 1)
# np.save(f"E_x_test_{n_src}.npy", E_x)

In [ ]:
# # Test of find_B_i()
# A_x = np.random.rand(3, 3, 3, 3)
# A_y = np.random.rand(3, 3, 3, 3)
# A_z = np.random.rand(3, 3, 3, 3)
# B_x = find_B_i([A_x, A_y, A_z], 1)
# print(f"A_x is {A_x}")
# print(f"A_y is {A_y}")
# print(f"A_z is {A_z}")
# print(f"B_x is {B_x}")

A_x is [[[[0.95307154 0.3869769  0.81464471]
   [0.92625786 0.48318043 0.09935442]
   [0.65977942 0.90902888 0.58763308]]

  [[0.78057705 0.02643401 0.23195566]
   [0.0043902  0.27951875 0.26583555]
   [0.57816752 0.5581665  0.43904707]]

  [[0.04150032 0.05817008 0.25300806]
   [0.72436458 0.44803979 0.56860463]
   [0.97810475 0.51435    0.91890006]]]


 [[[0.80400318 0.20275559 0.40467763]
   [0.17694077 0.95141178 0.59619766]
   [0.81901117 0.91792293 0.59824475]]

  [[0.61722773 0.00798019 0.67910435]
   [0.0899428  0.99873084 0.36475039]
   [0.55197416 0.25923369 0.59883985]]

  [[0.64473253 0.56582509 0.05307403]
   [0.85658534 0.97483502 0.62670432]
   [0.94130241 0.14511224 0.2623221 ]]]


 [[[0.11700951 0.27564612 0.96687832]
   [0.29951137 0.2632959  0.52804578]
   [0.60705034 0.98712846 0.89032515]]

  [[0.00273272 0.99889682 0.60025552]
   [0.42531668 0.14638614 0.30821971]
   [0.4559946  0.95861658 0.51063308]]

  [[0.12249758 0.98592893 0.11122947]
   [0.21745324 0.924399